# LMTA ALMP XML Export: Clean Start

**Notebook engine:** PySpark.

This notebook uses the PySpark runtime only for Microsoft Fabric integration and `notebookutils`. It does not use Spark DataFrames. Warehouse access, XML construction, XSD validation, and Lakehouse file output are ordinary Python.

Use only the new module name `lmta_almp_export.py`. Do not upload or import `almp_xml_export.py` in this notebook.


## Install Packages

In [1]:
%pip install lxml pyodbc pandas


StatementMeta(, a2426e1c-2a2a-4c01-b9de-620ac607772b, 7, Finished, Available, Finished, True)


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



If Fabric restarts the session after package installation, continue with the next cell.


## Import Modules

In [1]:
from pathlib import Path
import sys
import importlib
import json

MODULE_DIR = Path("/lakehouse/default/Files/LMTA/code/python")
MODULE_PATH = MODULE_DIR / "lmta_almp_export.py"

if not MODULE_PATH.exists():
    raise FileNotFoundError(f"Upload the new module to: {MODULE_PATH}")

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

# Unique module name prevents collision with all older almp_xml_export versions.
sys.modules.pop("lmta_almp_export", None)
importlib.invalidate_caches()
import lmta_almp_export

print("Loaded from:", lmta_almp_export.__file__)
print("Module version:", lmta_almp_export.MODULE_VERSION)
print("ExportConfig available:", hasattr(lmta_almp_export, "ExportConfig"))
print("test_connection available:", hasattr(lmta_almp_export, "test_connection"))
print("run_export available:", hasattr(lmta_almp_export, "run_export"))

ExportConfig = lmta_almp_export.ExportConfig
test_connection = lmta_almp_export.test_connection
run_export = lmta_almp_export.run_export


StatementMeta(, dc28579a-2f5f-4163-bf79-99d9eb5dade2, 3, Finished, Available, Finished, False)

Loaded from: /lakehouse/default/Files/LMTA/code/python/lmta_almp_export.py
Module version: 2026.11.06-clean-1
ExportConfig available: True
test_connection available: True
run_export available: True


## Create Configuration

In [2]:
# EDIT THESE VALUES ONLY
SUBMISSION_ID = 1001
SQL_SERVER = "esfe5ty3mtjebec6tiziwzcpvm-efk4ghrfe6uuzkqv3r573kj6be.datawarehouse.fabric.microsoft.com"
SQL_DATABASE = "LMTA"
PROGRAM_TYPE = "LMDA"       # LMDA or WDA
SEQUENCE_NUMBER = 1
MAIN_XSD_NAME = "ALMP.xsd"

# Production requirement: populate exact XSD xs:sequence order.
ELEMENT_ORDER = {
    # "submission": [...],
    # "sys_action": [...],
    # "client": [...],
    # "action_plan": [...],
    # "followup": [...],
    # "intervention": [...],
    # "support": [...],
}

config = ExportConfig(
    submission_id=SUBMISSION_ID,
    sql_server=SQL_SERVER,
    sql_database=SQL_DATABASE,
    program_type=PROGRAM_TYPE,
    sequence_number=SEQUENCE_NUMBER,
    main_xsd_name=MAIN_XSD_NAME,
    table_element_order=ELEMENT_ORDER,
)

print("Configuration created for submission:", config.submission_id)


StatementMeta(, dc28579a-2f5f-4163-bf79-99d9eb5dade2, 4, Finished, Available, Finished, False)

Configuration created for submission: 1001


## Test Connection

In [3]:
# Run this before XML generation.
connection_result = test_connection(config)
print(json.dumps(connection_result, indent=2, default=str))


StatementMeta(, dc28579a-2f5f-4163-bf79-99d9eb5dade2, 5, Finished, Available, Finished, False)

{
  "connected": true,
  "driver": "libmsodbcsql-18.4.so.1.1",
  "database": "LMTA",
  "latest_submission_id": 1001
}


## Run Export

**Required ICT_DL1 locations**

- Module: `Files/LMTA/code/python/lmta_almp_export.py`
- Main XSD: `Files/LMTA/reference/xsd/current/ALMP.xsd`
- Imported XSDs: same `current` folder
- Valid XML: `Files/LMTA/output/xml/valid/`
- Invalid XML: `Files/LMTA/output/xml/invalid/`
- Validation reports: `Files/LMTA/output/validation/`


In [4]:
# Run only after the connection test succeeds and SUBMISSION_ID exists.
result = run_export(config)
print(json.dumps(result, indent=2, default=str))


StatementMeta(, dc28579a-2f5f-4163-bf79-99d9eb5dade2, 6, Finished, Available, Finished, False)

/lakehouse/default/Files/LMTA/code/python/lmta_almp_export.py:215: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  frame = pd.read_sql_query(sql, connection, params=[cfg.submission_id])
/lakehouse/default/Files/LMTA/code/python/lmta_almp_export.py:215: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  frame = pd.read_sql_query(sql, connection, params=[cfg.submission_id])
/lakehouse/default/Files/LMTA/code/python/lmta_almp_export.py:215: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  frame = pd.read_sql_query(sql, connection, param

ValueError: XML validation failed with 6 error(s). Rejected XML: /lakehouse/default/Files/LMTA/output/xml/rejected/sklmda_001_20261106210846.xml; report: /lakehouse/default/Files/LMTA/output/validation/sklmda_001_20261106210846.xml.validation.json

## Troubleshoot the Errors

In [5]:
import json
from pathlib import Path

report_path = Path(
    "/lakehouse/default/Files/LMTA/output/validation/"
    "sklmda_001_20261106202911.xml.validation.json"
)

report = json.loads(
    report_path.read_text(encoding="utf-8")
)

print("Valid:", report["valid"])
print("XML:", report["xml_path"])
print("XSD:", report["xsd_path"])

print("\nVALIDATION ERRORS")
print("=" * 80)

for i, error in enumerate(report["errors"], start=1):
    print(f"\nError {i}")
    print("-" * 40)

    for key, value in error.items():
        print(f"{key}: {value}")

StatementMeta(, dc28579a-2f5f-4163-bf79-99d9eb5dade2, 7, Finished, Available, Finished, False)

Valid: False
XML: /lakehouse/default/Files/LMTA/output/xml/rejected/sklmda_001_20261106202911.xml
XSD: /lakehouse/default/Files/LMTA/reference/xsd/current/ALMP.xsd

VALIDATION ERRORS

Error 1
----------------------------------------
validation_type: XSD
severity: ERROR
line: 7
column: 0
domain: SCHEMASV
type: SCHEMAV_ELEMENT_CONTENT
message: Element 'sys_action_id': This element is not expected. Expected is one of ( socialInsuranceNumber, clientID ).

Error 2
----------------------------------------
validation_type: XSD
severity: ERROR
line: 14
column: 0
domain: SCHEMASV
type: SCHEMAV_ELEMENT_CONTENT
message: Element 'client_id': This element is not expected. Expected is one of ( clientID, socialInsuranceNumber ).

Error 3
----------------------------------------
validation_type: XSD
severity: ERROR
line: 765
column: 0
domain: SCHEMASV
type: SCHEMAV_ELEMENT_CONTENT
message: Element 'client_id': This element is not expected. Expected is one of ( clientID, socialInsuranceNumber ).

Error